# Week 39, Day 4: Efficient coding in space: why the retina has a surround

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w039/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The 1/f² spectrum

Write radial_power(img): the 2D power spectrum |FFT|², averaged within logarithmically spaced rings of radial frequency (bins given), returning the ring centres and mean powers. Then fit the slope of log power against log frequency.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1992); n = 128
fx = np.fft.fftfreq(n); FX, FY = np.meshgrid(fx, fx)
F = np.hypot(FX, FY); F[0, 0] = 1 / n
img = np.real(np.fft.ifft2(np.fft.fft2(r.normal(size=(n, n))) / F))     # 1/f amplitude, 1/f^2 power
bins = np.logspace(np.log10(2 / n), np.log10(0.5), 20)

def radial_power(img):
    P = np.abs(np.fft.fft2(img)) ** 2
    idx = np.digitize(F.ravel(), bins)
    power = ...
    centres = np.sqrt(bins[:-1] * bins[1:])
    return centres, power

fc, pr = radial_power(img)
slope = np.polyfit(np.log(fc), np.log(pr), 1)[0]
print(f"power spectrum slope {slope:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].imshow(img, cmap="gray"); ax[0].axis("off")
ax[1].loglog(fc, pr, "o-"); ax[1].set_xlabel("spatial frequency (cycles/pixel)"); ax[1].set_ylabel("power"); plt.show()

In [ ]:
import numpy as np
def test_slope():
    "the fitted log-log slope is between -2.2 and -1.8"
    assert -2.2 < slope < -1.8

test_slope()
print("All checks passed")

### Exercise: Centre-surround from efficiency

Write rf_profile(snr): build K(f) on the frequency grid F with S = snr / F² and N = 1 (set the zero-frequency term to 0), transform to space with the inverse FFT (fftshift to centre it), and return the profile along one row from the centre outwards. Then record each profile's most negative value relative to its peak, and the half-width of its centre.

In [ ]:
def rf_profile(snr):
    S, N = snr / F ** 2, 1.0
    K = ...
    K[0, 0] = 0
    k = np.real(np.fft.fftshift(np.fft.ifft2(K)))
    return k[n // 2, n // 2:]

surround, width = {}, {}
for snr in (1e-4, 1e-2, 1, 100):
    p = rf_profile(snr)
    surround[snr] = p.min() / p.max()
    width[snr] = int(np.argmax(p < p.max() / 2))
    plt.plot(p[:12] / p.max(), "o-", label=f"SNR {snr:g}")
    print(f"SNR {snr:8g}: surround {surround[snr]:+.3f} of peak, centre half-width {width[snr]} pixels")
plt.axhline(0, color="gray", lw=0.5); plt.xlabel("distance from centre (pixels)"); plt.ylabel("receptive field (relative)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_bright():
    "at high signal-to-noise ratio the receptive field has a surround below -10% of the centre's peak"
    assert surround[100] < -0.1

def test_dim():
    "at very low signal-to-noise ratio the surround is weaker than -3% and the centre is at least twice as wide"
    assert surround[1e-4] > -0.03 and width[1e-4] >= 2 * width[100]

test_bright()
test_dim()
print("All checks passed")

### Exercise: Removing spatial correlation

Filter img with the SNR 100 kernel (multiply their Fourier transforms) and compute the correlation coefficient between each pixel and the pixel 3 to its right (np.roll), before and after filtering.

In [ ]:
S100 = 100 / F ** 2; K100 = (S100 / (S100 + 1)) / np.sqrt(S100 + 1); K100[0, 0] = 0
filtered = np.real(np.fft.ifft2(np.fft.fft2(img) * K100))

def lag_corr(im, lag=3):
    ...

ac_raw, ac_filt = lag_corr(img), lag_corr(filtered)
print(f"correlation at 3 pixels: raw {ac_raw:.2f}, filtered {ac_filt:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].imshow(img, cmap="gray"); ax[1].imshow(filtered, cmap="gray")
for a, t in zip(ax, ["image", "after the efficient filter"]): a.set_title(t); a.axis("off")
plt.show()

In [ ]:
import numpy as np
def test_decor():
    "the correlation between pixels 3 apart falls from above 0.5 to below 0.1"
    assert ac_raw > 0.5 and abs(ac_filt) < 0.1

test_decor()
print("All checks passed")